In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Practice-null").getOrCreate()

In [3]:
from google.colab import files
files.upload()

Saving bookings.csv to bookings.csv


{'bookings.csv': b'booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode\r\n1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI\r\n1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card\r\n1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI\r\n1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash\r\n1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card\r\n1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,\r\n1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI\r\n1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card\r\n1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI\r\n1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card\r\n1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,\r\n1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI\r\n1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash\r\n1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card\r\n1015,Ravi Kumar,Delhi,Flight,SpiceJet,48

In [4]:
book_df=spark.read.csv("bookings.csv",header=True,inferSchema=True)

In [5]:
book_df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [6]:
book_df.count()

15

In [8]:
from pyspark.sql.functions import col
book_df.filter(col("city").isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [9]:
book_df.filter(col("provider").isNull()).show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|       Pending|        Cash|
|      1013|  Arjun Verma|  Chennai|      Flight|    NULL|         15000|     Cancelled|        Cash|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [10]:
book_df.filter(col("booking_amount").isNull()).show()

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|  city|service_type|     provider|booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|      1005|   Vikram Rao|Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|
|      1014|   Kavya Nair|Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [11]:
book_df.filter(col("booking_status").isNull()).show()

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|  provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|      1007|    Imran Ali|Pune|       Hotel|Budget Inn|          2200|          NULL|         UPI|
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [12]:
book_df.filter(col("payment_mode").isNull()).show()

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name| city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|      1006| Divya Sharma|Delhi|      Flight|        IndiGo|          5900|     Cancelled|        NULL|
|      1011|   Farhan Ali| NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [16]:
from pyspark.sql.functions import sum, col
null_counts=book_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in book_df.columns])
null_counts.show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [17]:
drop_all_null=book_df.na.drop()
drop_all_null.count()

6

In [18]:
drop_book_amount_null=book_df.na.drop(subset=["booking_amount"])
drop_book_amount_null.count()

13

In [19]:
result1=book_df.na.drop(subset=["customer_name","service_type","booking_amount"])
result1.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [21]:
fill_city=book_df.na.fill({"city":"Unknown"})
fill_city.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [22]:
fill_provider=book_df.na.fill({"provider":"Not Available"})
fill_provider.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [23]:
fill_payment_mode=book_df.na.fill({"payment_mode":"Not Provided"})
fill_payment_mode.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|Not Provided|
|      100

In [24]:
fill_booking_status=book_df.na.fill({"booking_status":"Unknown"})
fill_booking_status.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [25]:
fill_book_amount=book_df.na.fill({"booking_amount":0})
fill_book_amount.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [27]:
from pyspark.sql.functions import when
check_df=book_df.withColumn("data_quality_status",when((col("customer_name").isNull())|(col("service_type").isNull())|(col("booking_amount").isNull()),"Incomplete").otherwise("Complete"))
check_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Conf

In [28]:
check_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   12|
|         Incomplete|    3|
+-------------------+-----+



In [29]:
check_df.filter(col("data_quality_status")=="Incomplete").show()

+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|     provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+
|      1005|   Vikram Rao|   Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|         Incomplete|
|      1012|   Neha Singh|Hyderabad|        NULL|     Emirates|         28000|     Confirmed|         UPI|         Incomplete|
|      1014|   Kavya Nair|   Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|         Incomplete|
+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+



In [30]:
check_df.filter(col("data_quality_status")=="Complete").show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|           Complete|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Canc

In [31]:
book_df=book_df.withColumn("tax",book_df.booking_amount*0.5)
book_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|    tax|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI| 3250.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card| 2250.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI| 6000.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash| 3750.0|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|   NULL|
|      1006| Divya Sharma|    Delhi|      Flight|       

In [32]:
book_df=book_df.withColumn("total_amount",book_df.booking_amount+book_df.tax)
book_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|    tax|total_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI| 3250.0|      9750.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card| 2250.0|      6750.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI| 6000.0|     18000.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash| 3750.0|     11250.0|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|  

In [35]:
book_df.filter(book_df.booking_status=="Confirmed").select(sum("total_amount")).show()

+-----------------+
|sum(total_amount)|
+-----------------+
|         220950.0|
+-----------------+



In [36]:
fill_city.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [37]:
fill_service_type=book_df.na.fill({"service_type":"Not Provided"})
fill_service_type.groupBy("service_type").count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|       Hotel|    6|
|      Flight|    8|
|Not Provided|    1|
+------------+-----+



In [39]:
from pyspark.sql.functions import mean
replace_null=book_df.na.drop(subset=["booking_amount"])
replace_null.select(mean("booking_amount")).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 14253.846153846154|
+-------------------+



In [40]:
drop_null=book_df.na.drop(subset=["booking_amount"])
drop_null.select(mean("booking_amount")).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 14253.846153846154|
+-------------------+



In [42]:
clean_df=book_df.na.fill({"city":"Unknown","booking_status":"Unknown","booking_amount":0,"payment_mode":"Not Available","service_type":"Not Provided"})
clean_df.write.parquet("clean_bookings.parquet")

In [54]:
from google.colab import files
files.upload()

Saving customer.json to customer (1).json


{'customer (1).json': b'[\r\n{\r\n"customer_id": 1,\r\n"name": "Aarav Mehta",\r\n"city": "Hyderabad",\r\n"membership": "Gold",\r\n"contact": {\r\n"phone": "9876500011",\r\n"email": "aarav@mail.com"\r\n},\r\n"preferences": {\r\n"preferred_service": "Flight",\r\n"budget_range": "Medium"\r\n}\r\n},\r\n{\r\n"customer_id": 2,\r\n"name": "Sana Khan",\r\n"city": "Bangalore",\r\n"membership": "Silver",\r\n"contact": {\r\n"phone": null,\r\n"email": "sana@mail.com"\r\n},\r\n"preferences": {\r\n"preferred_service": "Hotel",\r\n"budget_range": null\r\n}\r\n},\r\n{\r\n"customer_id": 3,\r\n29. Compare both averages.\r\n30. Save clean data as clean_bookings.parquet.\r\n\r\n"name": "John Mathew",\r\n"city": null,\r\n"membership": "Gold",\r\n"contact": {\r\n"phone": "9876500013",\r\n"email": null\r\n},\r\n"preferences": {\r\n"preferred_service": "Flight",\r\n"budget_range": "High"\r\n}\r\n},\r\n{\r\n"customer_id": 4,\r\n"name": "Ayesha Begum",\r\n"city": "Hyderabad",\r\n"membership": null,\r\n"contact"

In [57]:
from pyspark.sql.functions import col
customers_df = spark.read.option("multiline","true").json("customer.json")
customers_df.show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|     NULL|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [58]:
flat_customers_df = customers_df.select(
"customer_id",
"name",
"city",
"membership",
col("contact.phone").alias("phone"),
col("contact.email").alias("email"),
col("preferences.preferred_service").alias("preferred_service"),
col("preferences.budget_range").alias("budget_range")
)
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [59]:
flat_customers_df.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- preferred_service: string (nullable = true)
 |-- budget_range: string (nullable = true)



In [61]:
flat_customers_df.select("name","city","phone","email").show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [67]:
flat_customers_df.filter(col("city").isNull()).show()

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [68]:
flat_customers_df.filter(col("phone").isNull()).show()

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|          2| Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [69]:
flat_customers_df.filter(col("email").isNull()).show()

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|       name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|          3|John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5| Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [70]:
flat_customers_df.filter(col("membership").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [71]:
flat_customers_df.filter(col("preferred_service").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [72]:
flat_customers_df.filter(col("budget_range").isNull()).show()

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|     name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|          2|Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [76]:
from pyspark.sql.functions import sum, col, lit
null_counts_customers_per_column = flat_customers_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in flat_customers_df.columns])
for c_name in null_counts_customers_per_column.columns:
    total_sum_expression += col(c_name)

total_null_values_df = null_counts_customers_per_column.select(total_sum_expression.alias("total_null_values"))
print("Total null values in the DataFrame (sum of all nulls across all columns):")
total_null_values_df.show()

Total null values in the DataFrame (sum of all nulls across all columns):
+-----------------+
|total_null_values|
+-----------------+
|               16|
+-----------------+



In [77]:
fill_city=customers_df.na.fill({"city":"Unknown"})
fill_city.show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|  Unknown|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [79]:
fill_membership=customers_df.na.fill({"membership":"Standard"})
fill_membership.show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|     NULL|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|  Standard|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [83]:
fill_phone=flat_customers_df.na.fill({"phone":"Not Available"})
fill_phone.show()

+-----------+------------+---------+----------+-------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|        phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+-------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|   9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Available|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|   9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|   9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Available|           NULL|           Flight|        High|
+-----------+------------+---------+----------+-------------+---------------+-----------------+------------+



In [84]:
fill_email=flat_customers_df.na.fill({"email":"Not Provided"})
fill_email.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [85]:
fill_service=flat_customers_df.na.fill({"preferred_service":"Not Selected"})
fill_service.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [86]:
fill_budget_range=flat_customers_df.na.fill({"budget_range":"Unknown"})
fill_budget_range.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [87]:
flat_customers_df=flat_customers_df.withColumn("data_quality_status",when((col("city").isNull())|(col("phone").isNull())|(col("email").isNull())|(col("membership").isNull())|(col("preferred_service").isNull()),"Incomplete").otherwise("Complete"))
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|data_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|           Complete|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|         Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|         Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|         Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|         Inco

In [88]:
flat_customers_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    1|
|         Incomplete|    4|
+-------------------+-----+



In [89]:
fill_membership.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [90]:
fill_service.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [91]:
flat_customers_df.write.parquet("customer_flat.parquet")

In [92]:
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|data_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|           Complete|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|         Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|         Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|         Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|         Inco

In [95]:
flat_customers_df=flat_customers_df.na.fill({"city":"Unknown","phone":"Not Available","email":"Not Provided","membership":"Standard","preferred_service":"Not Selected"})
flat_customers_df.show()

+-----------+------------+---------+----------+-------------+---------------+-----------------+------------+-------------------+
|customer_id|        name|     city|membership|        phone|          email|preferred_service|budget_range|data_quality_status|
+-----------+------------+---------+----------+-------------+---------------+-----------------+------------+-------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|   9876500011| aarav@mail.com|           Flight|      Medium|           Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Available|  sana@mail.com|            Hotel|        NULL|         Incomplete|
|          3| John Mathew|  Unknown|      Gold|   9876500013|   Not Provided|           Flight|        High|         Incomplete|
|          4|Ayesha Begum|Hyderabad|  Standard|   9876500014|ayesha@mail.com|     Not Selected|         Low|         Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Available|   Not Provided|           Flight|  

In [97]:
flat_customers_df.write.csv("clean_customers.csv", header=True, mode="overwrite")

In [100]:
customers_df.filter(col("contact.phone").isNull()).show()

+---------+--------------------+-----------+----------+----------+--------------+
|     city|             contact|customer_id|membership|      name|   preferences|
+---------+--------------------+-----------+----------+----------+--------------+
|Bangalore|{sana@mail.com, N...|          2|    Silver| Sana Khan| {NULL, Hotel}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|Vikram Rao|{High, Flight}|
+---------+--------------------+-----------+----------+----------+--------------+



In [103]:
flat_customers_df.filter(col("preferred_service").isNull() | col("budget_range").isNull()).show()

+-----------+---------+---------+----------+-------------+-------------+-----------------+------------+-------------------+
|customer_id|     name|     city|membership|        phone|        email|preferred_service|budget_range|data_quality_status|
+-----------+---------+---------+----------+-------------+-------------+-----------------+------------+-------------------+
|          2|Sana Khan|Bangalore|    Silver|Not Available|sana@mail.com|            Hotel|        NULL|         Incomplete|
+-----------+---------+---------+----------+-------------+-------------+-----------------+------------+-------------------+

